In [ ]:
from google.colab import drive, userdata
import os, sys

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/DataScience/molecular-ai-agent'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.append(PROJECT_DIR)

In [ ]:
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install rdkit -q
!pip install transformers peft accelerate bitsandbytes -q
!pip install unsloth -q
!pip install gradio -q

print("All packages ready.")

In [ ]:
%%writefile /content/drive/MyDrive/DataScience/molecular-ai-agent/src/models/agent.py
from rdkit import Chem
import torch
from torch_geometric.data import Data
from src.utils.mol_utils import smiles_to_features, get_basic_descriptors

class MolecularAgent:

  MOLECULE_KEYWORDS = [
        'smiles', 'molecule', 'predict', 'property',
        'structure', 'compound', 'formula', 'atoms'
    ]

  CHEMISTRY_KEYWORDS = [
        'homo', 'lumo', 'gap', 'dipole', 'energy',
        'orbital', 'electron', 'aromatic', 'bond',
        'explain', 'what is', 'why', 'how does',
        'what does', 'define', 'meaning'
    ]

  def __init__(self, gnn_model = None, llm_pipeline = None, device = 'cpu'):
    self.gnn = gnn_model
    self.llm = llm_pipeline
    self.device = device

  def classify_input(self, user_input: str) -> str:
    text = user_input.lower().strip()
    has_smiles_chars = any(c in user_input for c in ['=', '#', '[', ']'])
    has_mol_keyword = any(kw in text for kw in self.MOLECULE_KEYWORDS)
    if has_smiles_chars or has_mol_keyword:
      return 'gnn'
    has_chem_keyword = any(kw in text for kw in self.CHEMISTRY_KEYWORDS)
    if has_chem_keyword:
      return 'llm'
    return 'llm'

  def extract_smiles(self, user_input: str):
    words = user_input.split()
    for word in words:
      word = word.strip('.,?!')
      if any(c in word for c in ['=', '#', '(', ')', '[', ']']):
        if Chem.MolFromSmiles(word) is not None:
          return word
    return None

  def predict_molecule(self, smiles: str) -> str:
    node_features, edge_index, valid = smiles_to_features(smiles)
    if not valid:
      return {'error': 'Invalid SMILES string.'}

    x = torch.tensor(node_features, dtype = torch.float)
    edge_index = torch.tensor(edge_index, dtype = torch.long).t().contiguous()
    batch = torch.zeros(x.shape[0], dtype = torch.long)

    data = Data(x=x, edge_index = edge_index, batch = batch).to(self.device)

    self.gnn.eval()
    with torch.no_grad():
      prediction = self.gnn(data).item()

    descriptors = get_basic_descriptors(smiles)

    return {
        'smiles': smiles,
        'homo_lumo_gap': round(prediction, 4),
        'unit': 'Hartree',
        'descriptors': descriptors
    }

  def ask_llm(self, question: str) -> str:
    if self.llm is None:
      return "LLM  not loaded."
    model, tokenizer = self.llm
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    prompt = f"""### Instruction:
  {question}

  ### Response:
  """
    inputs = tokenizer(prompt, return_tensors = 'pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        max_length=None,
        temperature=0.0,
        top_p=1.0,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens = True)
    return response.split('### Response:')[-1].strip()


  def run(self, user_input: str) -> dict:
    route = self.classify_input(user_input)

    if route == 'gnn':
      smiles = self.extract_smiles(user_input)
      if smiles is None:
        return {
            'route': 'gnn',
            'error': 'No valid SMILES found.',
            'suggestion': 'Example: CC(=O)O for acetic acid'

        }
      if self.gnn is None:
        return {'route': 'gnn', 'error': 'GNN model not loaded.'}
      result = self.predict_molecule(smiles)
      result['route'] = 'gnn'
      return result
    else:
      response = self.ask_llm(user_input)
      return {
          'route': 'llm',
          'question': user_input,
          'answer': response
      }


Load GNN model

In [ ]:
import torch, sys
sys.path.append(PROJECT_DIR)

# Clear cache
mods_to_remove = [k for k in sys.modules if 'src' in k]
for m in mods_to_remove:
    del sys.modules[m]

from src.models.gnn import MolecularGNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gnn_model = MolecularGNN(
    node_features=11,
    hidden_dim=128,
    num_layers=4,
    dropout=0.1
).to(device)

gnn_model.load_state_dict(
    torch.load(f'{PROJECT_DIR}/models/best_gnn.pt', map_location=device)
)
gnn_model.eval()

print(f"GNN loaded on: {device}")
print(f"Parameters   : {sum(p.numel() for p in gnn_model.parameters()):,}")

Load fine-tuned LLM

In [ ]:
from unsloth import FastLanguageModel

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = f'{PROJECT_DIR}/models/llm_finetuned',
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

print("LLM loaded successfully.")
llm_pipeline = (llm_model, tokenizer)

Initialize agent and test

In [ ]:
from src.models.agent import MolecularAgent

agent = MolecularAgent(
    gnn_model    = gnn_model,
    llm_pipeline = llm_pipeline,
    device       = device
)

# Test 1 — molecule prediction
result1 = agent.run("predict CC(=O)O")
print("Test 1 — Molecule prediction:")
print(f"  Route: {result1['route']}")
print(f"  SMILES: {result1.get('smiles')}")
print(f"  HOMO-LUMO gap: {result1.get('homo_lumo_gap')} Hartree")
print(f"  Descriptors: {result1.get('descriptors')}")
print()

# Test 2 — chemistry question
result2 = agent.run("What is the HOMO-LUMO gap?")
print("Test 2 — Chemistry question:")
print(f"  Route  : {result2['route']}")
print(f"  Answer : {result2.get('answer')[:200]}...")

In [ ]:
from rdkit import Chem
# from rdkit.Chem.rdmolops import GetthAtomNeighbors
from rdkit.Chem import Descriptors

def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, False

    # Atom features (11 features)
    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),           # 1. Atomic number (int)
            atom.GetDegree(),              # 2. Degree (number of bonds) (int)
            atom.GetFormalCharge(),        # 3. Formal charge (int)
            atom.GetImplicitValence(),     # 4. Implicit valence (int)
            int(atom.GetIsAromatic()),     # 5. Is aromatic (boolean as int)
            atom.GetNumRadicalElectrons(), # 6. Number of radical electrons (int)
            atom.GetHybridization(),       # 7. Hybridization (enum value)
            atom.GetTotalNumHs(),          # 8. Total number of Hs (int)
            int(atom.IsInRing()),          # 9. Is in a ring (boolean as int)
            atom.GetExplicitValence(),     # 10. Explicit valence (int)
            atom.GetChiralTag()            # 11. Chiral tag (enum value, added this to make 11 features)
        ]
        node_features.append(features)

    # Edge features
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i]) # Add reverse edge for undirected graph

    return node_features, edge_index, True

def get_basic_descriptors(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'error': 'Invalid SMILES string for descriptors.'}

    return {
        'molecular_weight': Descriptors.MolWt(mol),
        'num_atoms': mol.GetNumAtoms(),
        'num_rings': Chem.GetSSSR(mol),
        'logP': Descriptors.MolLogP(mol),
        'num_h_donors': Descriptors.NumHDonors(mol),
        'num_h_acceptors': Descriptors.NumHAcceptors(mol),
    }


In [ ]:
%%writefile /content/drive/MyDrive/DataScience/molecular-ai-agent/src/app.py
import gradio as gr
import os

sys.path.append('/content/drive/MyDrive/DataScience/molecular-ai-agent')

from src.utils.mol_utils import draw_molecule, get_basic_descriptors

def process_input(user_input, agent):
  if not user_input.strip():
    return "Please enter a SMILES string or chemistry question."

  result = agent.run(user_input)

  if result.get('error'):
    return result['error'], None, ""

  if result['route'] == 'gnn':
    smiles = result.get('smiles', '')
    gap = result.get('homo_lumo_gap', 'N/A')
    desc = result.get('descriptors', {})

    # Draw molecule
    img = draw_molecule(smiles)

    # Format prediction output
    output = f"""Molecule Analysis
─────────────────────────────
SMILES         : {smiles}
HOMO-LUMO Gap  : {gap} eV

Molecular Properties
─────────────────────────────
Molecular Weight : {desc.get('molecular_weight', 'N/A')} g/mol
Num Atoms        : {desc.get('num_atoms', 'N/A')}
Num Rings        : {desc.get('num_rings', 'N/A')}
LogP             : {desc.get('logP', 'N/A')}
H-Bond Donors    : {desc.get('num_h_donors', 'N/A')}
H-Bond Acceptors : {desc.get('num_h_acceptors', 'N/A')}
"""
    if gap > 8:
          interp = f"Gap of {round(gap,2)} eV — Very stable, chemically inert. Excellent insulator."
    elif gap > 6:
          interp = f"Gap of {round(gap,2)} eV — Stable molecule, low reactivity. Good insulator."
    elif gap > 4:
          interp = f"Gap of {round(gap,2)} eV — Moderate stability. Typical drug-like molecule."
    elif gap > 2:
          interp = f"Gap of {round(gap,2)} eV — Moderately reactive. Potential semiconductor material."
    else:
          interp = f"Gap of {round(gap,2)} eV — Highly reactive. Potential organic conductor or dye."
    return output, img, interp

  else:
    return result.get('answer', 'No response.'), None, ""

def create_app(agent):
  with gr.Blocks(title = 'QuantumMind - Molecular AI Agent') as app:

    gr.Markdown("""
    # QuantumMind AI
    Enter a SMILES string to predict molecular properties, or ask a chemistry question.
    """)

    with gr.Row():
      with gr.Column(scale = 2):
        user_input = gr.Textbox(
            label = 'Input',
            placeholder = 'Enter SMILES (e.g. CC(=O)O) or chemistry question',
            lines = 2
        )
        submit_btn = gr.Button("Analyse", variant = "primary")

      with gr.Column(scale=1):
        mol_image = gr.Image(label = 'Molecule Structure')

    with gr.Row():
      output_text = gr.Textbox(label = 'Prediction / Answer', lines = 12)

    with gr.Row():
      interpretation = gr.Textbox(label = 'Interpretation', lines = 2)

    submit_btn.click(
        fn = lambda x: process_input(x, agent),
        inputs = [user_input],
        outputs = [output_text, mol_image, interpretation]
    )

  return app

Launch Gradio app

In [ ]:
import sys
sys.path.append(PROJECT_DIR)

if 'src.app' in sys.modules:
  del sys.modules['src.app']

exec(open(f'{PROJECT_DIR}/src/app.py').read())

app = create_app(agent)
app.launch(share = True, debug = False)